In [28]:
# !pip install llama-index-llms-cleanlab llama-index llama-index-embeddings-huggingface

Setup LLM

In [22]:
import os
from dotenv import load_dotenv
load_dotenv()

from llama_index.llms.cleanlab import CleanlabTLM


options = {
    "model": "gpt-4o",
    "max_tokens": 256,
    "log": ["explanation"],
}

llm = CleanlabTLM(api_key=os.environ["CLEANLAB_API_KEY"], options=options)

In [23]:
response = llm.complete("What is NVIDIA's ticker symbol?")
print(response)

NVIDIA's ticker symbol is NVDA.


In [24]:
response.additional_kwargs

{'trustworthiness_score': 0.9885545474223644,
 'explanation': 'Did not find a reason to doubt trustworthiness.'}

In [25]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

from utils import (
    setup_trustworthiness_handler,
    display_response
)


Settings.llm = llm

In [26]:
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

In [27]:
from typing import Dict, List, ClassVar
from llama_index.core.instrumentation.events import BaseEvent
from llama_index.core.instrumentation.event_handlers import BaseEventHandler
from llama_index.core.instrumentation import get_dispatcher
from llama_index.core.instrumentation.events.llm import LLMCompletionEndEvent


class GetTrustworthinessScoreAndReasoning(BaseEventHandler):
    events: ClassVar[List[BaseEvent]] = []
    trustworthiness_score: float = 0.0
    reasoning: str = ""

    @classmethod
    def class_name(cls) -> str:
        """Class name."""
        return "GetTrustworthinessScoreAndReasoning"

    def handle(self, event: BaseEvent) -> Dict:
        if isinstance(event, LLMCompletionEndEvent):
            self.trustworthiness_score = event.response.additional_kwargs[
                "trustworthiness_score"
            ]
            self.reasoning = event.response.additional_kwargs[
                "explanation"
            ]
            self.events.append(event)


# Root dispatcher
root_dispatcher = get_dispatcher()

# Register event handler
event_handler = GetTrustworthinessScoreAndReasoning()
root_dispatcher.add_event_handler(event_handler)

def display_response(response):
    response_str = response.response
    trustworthiness_score = event_handler.trustworthiness_score
    reasoning = event_handler.reasoning
    print(f"Response: {response_str}")
    print(f"Trustworthiness score: {round(trustworthiness_score, 2)}")
    print(f"Reasoning: {reasoning}")


In [22]:
# Optional: Define `display_response` helper function


# This method presents formatted responses from our TLM-based RAG pipeline. It parses the output to display both the text response itself and the corresponding trustworthiness score.


In [28]:
# LlamaParse PDF reader for PDF Parsing
from llama_parse import LlamaParse

documents = LlamaParse(result_type="markdown").load_data(
    "/Users/amarnagargoje/Documents/genai-projects/Agentic RAG/trustworthy-RAG/docs/dspy.pdf"
)


Started parsing the file under job_id ff4a2e97-65e3-4cbc-b616-3da0b85625b8


In [29]:
index = VectorStoreIndex.from_documents(documents)

In [30]:
query_engine = index.as_query_engine()

In [31]:
response = query_engine.query("What is is DSPy and mentions its significane?")
print(response)

DSPy is a framework designed to abstract and optimize the process of prompting language models (LMs). It introduces the concept of natural language signatures, which are declarative specifications that describe what a text transformation needs to do, rather than how to prompt a specific LM to achieve that behavior. These signatures are used to create modules that can be compiled into self-improving and pipeline-adaptive prompts or fine-tunes, reducing the need for brittle string manipulation in user programs.

The significance of DSPy lies in its ability to tackle the fundamental challenges of prompt engineering by providing a structured framework that automatically bootstraps prompts. This contrasts with existing libraries like LangChain and LlamaIndex, which rely heavily on manual prompt engineering. DSPy aims to help researchers and practitioners build new LM pipelines quickly and achieve high quality through automatic compilation and self-improvement, rather than relying on pre-pac

In [32]:
display_response(response)

Response: DSPy is a framework designed to abstract and optimize the process of prompting language models (LMs). It introduces the concept of natural language signatures, which are declarative specifications that describe what a text transformation needs to do, rather than how to prompt a specific LM to achieve that behavior. These signatures are used to create modules that can be compiled into self-improving and pipeline-adaptive prompts or fine-tunes, reducing the need for brittle string manipulation in user programs.

The significance of DSPy lies in its ability to tackle the fundamental challenges of prompt engineering by providing a structured framework that automatically bootstraps prompts. This contrasts with existing libraries like LangChain and LlamaIndex, which rely heavily on manual prompt engineering. DSPy aims to help researchers and practitioners build new LM pipelines quickly and achieve high quality through automatic compilation and self-improvement, rather than relying 